# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR² Dataset Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll show how to load Croissant metadata, review record sets and fields (using their `@id`), extract records into pandas DataFrames, and perform basic exploratory data analysis and visualization.

### Dataset Source
The dataset is described by a Croissant schema:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Install the `mlcroissant` library if not already installed
!pip install mlcroissant

## 1. Data Loading

We load the FAIR² dataset and its metadata using `mlcroissant`. The metadata summarizes the dataset structure, including record sets, fields, and column definitions. We'll use the `@id` fields to refer to and explore these components.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package with Croissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Next, let's review the available record sets, fields, and their `@id`s. As per the Croissant standard, these define the logical table layout. We'll display the `@id` for each record set, and for each record set, the available fields and their `@id`s.

In [ ]:
# Enumerate the available record sets via their `@id`
print("List of available record sets (@id):")
record_sets = list(dataset.record_sets.keys())
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"- {rs_id} (name: {record_set.name})")
    # List all available fields in this record set with their @id
    print("    Fields:")
    for field_id, field in record_set.fields.items():
        print(f"        - {field_id} (name: {field.name}, dataType: {field.data_type})")
    print("")

## 3. Data Extraction

Let's extract the data from the dataset. We'll load all records from each record set using their `@id` and organize them into pandas DataFrames, referenced by record set `@id`.

Select the record set(s) of interest by their `@id`. Typically, for tabular datasets like this, there will be one main record set.

In [ ]:
# Define the record sets you wish to extract (by @id)
record_set_ids = list(dataset.record_sets.keys())
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    df = dataframes[rs_id]
    print(f"\nColumns for record set {rs_id}:")
    print(df.columns.tolist())

# For demonstration, show first 5 rows of the first record set
main_rs_id = record_set_ids[0]
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Now, let's do some initial data exploration! We'll select a numeric field (such as age field or an interval field), filter values, normalize, and group by a categorical variable, all referenced by `@id`.

In [ ]:
# Identify a numeric field to analyze (please adjust based on column names in previous output)
# Example candidates: 'schema:age', 'cr:interval_between_diagnoses', etc.

df = dataframes[main_rs_id]
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'biufc']
print("Numeric fields found:", numeric_candidates)

if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Use first numeric field found
else:
    # Fallback field if there is no numeric column autodetected
    numeric_field_id = df.columns[0]

print(f"\nUsing numeric field for EDA: {numeric_field_id}")

# Filter records where the numeric field value is above a threshold; adjust threshold as needed
threshold = df[numeric_field_id].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None

if threshold is not None:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} above 75th percentile ({threshold}):")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field (string/object type), e.g., sex or anatomical_site
    group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
    if group_candidates:
        group_field_id = group_candidates[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped filtered data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df)
    else:
        print("No suitable categorical field found for grouping.")
else:
    print(f"Field {numeric_field_id} is not numeric; cannot filter/normalize.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and the relationship between the numeric and group fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the selected numeric field
if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True, bins=15, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in main record set")
    plt.show()

    # If a group/categorical field was found, show boxplot
    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print(f"No numeric field {numeric_field_id} found for visualization.")

## 6. Conclusion

- We loaded the FAIR² CRC survivor dataset using the Croissant schema and `mlcroissant` library.
- Using record set and field `@id`s, we extracted the main tabular data and inspected its columns.
- Basic EDA included filtering, normalization, grouping, and visualizing distributions for key numeric variables.
- This workflow provides a reproducible foundation for robust clinical research data exploration in Python, based on the Croissant data standard.